In [ ]:
# =======================================================================
# CÉLULA 1 — Instalação de dependências
# =======================================================================
!pip uninstall torchvision -y
!pip install -q -U transformers datasets peft trl accelerate
!pip install -q rouge-score nltk evaluate
!pip install -q google-generativeai
!pip install -q "torchao>=0.16.0"

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 146.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 53.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 110.7 MB/s eta 0:00:00


In [ ]:
# =======================================================================
# CÉLULA 2 — Imports
# =======================================================================
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import evaluate
import nltk
import google.generativeai as genai
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, StoppingCriteria, StoppingCriteriaList
from tqdm import tqdm

nltk.download("wordnet")
nltk.download("punkt")
nltk.download("punkt_tab")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# =======================================================================
# CÉLULA 3 — Configurações gerais
# =======================================================================
GEMINI_API_KEY = "sua_chave_aqui"
genai.configure(api_key=GEMINI_API_KEY)

MODEL_ID = "microsoft/bitnet-b1.58-2B-4T-bf16"
OUTPUT_DIR = "./avaliacao_mediasum_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_AMOSTRAS_AUTO = 300       # Mude para 300 quando for rodar completo
N_AMOSTRAS_MANUAL = 40      # Mude para 40 quando for rodar completo

SEED = 42

In [ ]:
# =======================================================================
# CÉLULA 4 — Download e extração do MediaSum
# =======================================================================
from huggingface_hub import hf_hub_download
import zipfile
import json

print("Baixando test_data.zip do MediaSum...")
zip_path = hf_hub_download(
    repo_id="ccdv/mediasum",
    filename="test_data.zip",
    repo_type="dataset"
)
extract_path = "./mediasum_data/test"
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)
print("-> Extraído com sucesso!")

Baixando test_data.zip do MediaSum...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


test_data.zip:   0%|          | 0.00/33.9M [00:00<?, ?B/s]

-> Extraído com sucesso!


In [ ]:
# =======================================================================
# CÉLULA 5 — Carregamento do dataset com formato Speaker: fala
# =======================================================================
print("Carregando MediaSum test set...")

dados_test = []
with open("./mediasum_data/test/test_data.txt", "r") as f:
    for linha in f:
        item = json.loads(linha)

        # Formato Speaker: fala
        texto = "\n".join([
            f"{speaker}: {fala}"
            for speaker, fala in zip(item["speaker"], item["utt"])
        ])
        resumo = item["summary"]

        if not texto or not resumo:
            continue

        # Filtro ajustado para 1700 palavras (considera prefixos dos speakers)
        quantidade_palavras = len(texto.split())
        if quantidade_palavras <= 1700:
            dados_test.append({"document": texto, "summary": resumo})

test_set_completo = Dataset.from_list(dados_test)

if N_AMOSTRAS_AUTO is not None:
    test_set = test_set_completo.shuffle(seed=SEED).select(range(N_AMOSTRAS_AUTO))
else:
    test_set = test_set_completo

print(f"-> {len(test_set_completo)} instâncias no test set completo")
print(f"-> {len(test_set)} instâncias carregadas para avaliação")
print(f"\nAmostra do primeiro exemplo:")
print(f"DOCUMENTO (primeiros 300 chars):\n{test_set[0]['document'][:300]}")
print(f"\nSUMMARY: {test_set[0]['summary']}")

Carregando MediaSum test set...
-> 6528 instâncias no test set completo
-> 300 instâncias carregadas para avaliação

Amostra do primeiro exemplo:
DOCUMENTO (primeiros 300 chars):
M. O'BRIEN: You know, when I played "Monopoly," I never liked having the utilities. Did you like having the utilities?
ANDY SERWER, "FORTUNE": Steady dividend. Steady.
M. O'BRIEN: It would apparently be a good buy right now, right, Andy Serwer?
SERWER: Absolutely. I'll tell you what's going on right

SUMMARY: Utility Usage High Due to Heat


In [ ]:
# =======================================================================
# CÉLULA 6 — Função de inferência
# =======================================================================
class StopOnToken(StoppingCriteria):
    def __init__(self, stop_token_ids):
        self.stop_token_ids = stop_token_ids

    def __call__(self, input_ids, scores, **kwargs):
        for stop_id in self.stop_token_ids:
            if input_ids[0][-1] == stop_id:
                return True
        return False

# Template — altere conforme necessário
template = (
    "### Instruction:\nWrite a short news headline summarizing the main topic of this interview in 16 words or less.\n\n"
    "### Interview Transcript:\n{document}\n\n"
    "### Headline:\n"
)


def gerar_resumos(model, tokenizer, dataset, modo="baseline"):
    resumos_gerados = []
    resumos_referencia = []
    dialogos = []

    # stop_sequences = ["#####", "####", "###", "### Instruction", "### Interview",
    #                   "def ", "```", "**Note:**", "---"]

    stop_sequences = ["#####", "####", "###", "### Instruction", "### Interview",
                      "def ", "```", "Note:", "---"]

    stop_ids = []
    for seq in stop_sequences:
        ids = tokenizer.encode(seq, add_special_tokens=False)
        if ids:
            stop_ids.append(ids[0])
    stopping_criteria = StoppingCriteriaList([StopOnToken(stop_ids)])

    for item in tqdm(dataset, desc=f"Gerando resumos ({modo})"):
        prompt = template.format(document=item["document"])

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.3,
                stopping_criteria=stopping_criteria
            )

        gerado = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()

        for seq in stop_sequences:
            if seq in gerado:
                gerado = gerado[:gerado.index(seq)].strip()

        resumos_gerados.append(gerado)
        resumos_referencia.append(item["summary"])
        dialogos.append(item["document"])

    return dialogos, resumos_referencia, resumos_gerados

In [ ]:
# =======================================================================
# CÉLULA 7 — Funções de métricas automáticas
# =======================================================================
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")

def calcular_bertscore_manual(referencias, gerados):
    referencias = [str(r) if r is not None else "" for r in referencias]
    gerados = [str(g) if g is not None else "" for g in gerados]

    model_name = "roberta-large"
    tokenizer_bert = AutoTokenizer.from_pretrained(model_name)
    model_bert = AutoModel.from_pretrained(model_name).to("cuda")
    model_bert.eval()

    scores_p, scores_r, scores_f1 = [], [], []

    for ref, gen in tqdm(zip(referencias, gerados), total=len(referencias), desc="BERTScore"):
        if not ref.strip() or not gen.strip():
            scores_p.append(0.0)
            scores_r.append(0.0)
            scores_f1.append(0.0)
            continue

        with torch.no_grad():
            ref_tokens = tokenizer_bert(ref, return_tensors="pt", truncation=True, max_length=512).to("cuda")
            gen_tokens = tokenizer_bert(gen, return_tensors="pt", truncation=True, max_length=512).to("cuda")

            ref_emb = model_bert(**ref_tokens).last_hidden_state.squeeze(0)
            gen_emb = model_bert(**gen_tokens).last_hidden_state.squeeze(0)

            ref_emb = ref_emb / ref_emb.norm(dim=-1, keepdim=True)
            gen_emb = gen_emb / gen_emb.norm(dim=-1, keepdim=True)

            sim = torch.matmul(gen_emb, ref_emb.T)

            P = sim.max(dim=1).values.mean().item()
            R = sim.max(dim=0).values.mean().item()
            F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0

            scores_p.append(P)
            scores_r.append(R)
            scores_f1.append(F1)

    del model_bert
    torch.cuda.empty_cache()

    return torch.tensor(scores_p), torch.tensor(scores_r), torch.tensor(scores_f1)

def calcular_metricas(referencias, gerados, prefixo=""):
    print(f"\nCalculando métricas {prefixo}...")

    gerados_clean = [g if g.strip() else "empty" for g in gerados]

    rouge_result = rouge.compute(
        predictions=gerados_clean,
        references=referencias,
        use_stemmer=True
    )
    bleu_result = bleu.compute(
        predictions=gerados_clean,
        references=[[r] for r in referencias]
    )
    meteor_result = meteor.compute(
        predictions=gerados_clean,
        references=referencias
    )

    P, R, F1 = calcular_bertscore_manual(referencias, gerados)

    resultados = {
        "ROUGE-1": round(rouge_result["rouge1"], 4),
        "ROUGE-2": round(rouge_result["rouge2"], 4),
        "ROUGE-L": round(rouge_result["rougeL"], 4),
        "BLEU": round(bleu_result["bleu"], 4),
        "METEOR": round(meteor_result["meteor"], 4),
        "BERTScore-P": round(P.mean().item(), 4),
        "BERTScore-R": round(R.mean().item(), 4),
        "BERTScore-F1": round(F1.mean().item(), 4),
    }

    return resultados

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
# =======================================================================
# CÉLULA 8 — Seleção estratificada das amostras
# =======================================================================
def selecionar_amostras_estratificadas(df, n=40, seed=SEED):
    df = df.copy()
    df_sorted = df.sort_values("rouge_l_individual").reset_index(drop=True)
    n_total = len(df_sorted)
    n_por_faixa = n // 3

    baixo = df_sorted.iloc[:n_total//3].sample(n=n_por_faixa, random_state=seed)
    medio = df_sorted.iloc[n_total//3:2*n_total//3].sample(n=n_por_faixa, random_state=seed)
    alto = df_sorted.iloc[2*n_total//3:].sample(n=n - 2*n_por_faixa, random_state=seed)

    amostras = pd.concat([baixo, medio, alto]).sample(frac=1, random_state=seed)
    return amostras

In [ ]:
# =======================================================================
# CÉLULA 9 — G-Eval com Gemini (desativado por ora)
# =======================================================================
import time

def geval_gemini(dialogo, resumo_referencia, resumo_gerado):
    prompt = f"""You are an expert evaluator of text summarization quality.
You will evaluate a generated summary based on a dialogue and a reference summary.
Score each dimension from 1 to 5, where 1 is very poor and 5 is excellent.

DIALOGUE:
{dialogo}

REFERENCE SUMMARY:
{resumo_referencia}

GENERATED SUMMARY:
{resumo_gerado}

Evaluate the generated summary on these 4 dimensions:
- Fluency (1-5): Is the text grammatically correct and natural to read?
- Coherence (1-5): Are ideas logically organized and well connected?
- Consistency (1-5): Does the summary avoid contradicting or hallucinating information from the dialogue?
- Relevance (1-5): Does the summary capture the most important information?

Respond ONLY with a JSON object in this exact format, no extra text:
{{"fluency": <score>, "coherence": <score>, "consistency": <score>, "relevance": <score>}}"""

    for tentativa in range(5):
        try:
            gemini_model = genai.GenerativeModel("gemini-2.0-flash-lite")
            response = gemini_model.generate_content(prompt)
            texto = response.text.strip()
            texto = texto.replace("```json", "").replace("```", "").strip()
            scores = eval(texto)
            return scores
        except Exception as e:
            if "429" in str(e):
                espera = 30 * (tentativa + 1)
                print(f"Quota atingida (tentativa {tentativa+1}/5), aguardando {espera}s...")
                time.sleep(espera)
            else:
                print(f"Erro no G-Eval: {e}")
                return {"fluency": None, "coherence": None, "consistency": None, "relevance": None}

    print("G-Eval falhou após 5 tentativas.")
    return {"fluency": None, "coherence": None, "consistency": None, "relevance": None}


def calcular_geval(amostras_df):
    print("\nRodando G-Eval com Gemini nas amostras selecionadas...")
    scores = []
    for _, row in tqdm(amostras_df.iterrows(), total=len(amostras_df)):
        score = geval_gemini(
            row["dialogo"],
            row["resumo_referencia"],
            row["resumo_gerado"]
        )
        scores.append(score)

    amostras_df["geval_fluency"] = [s["fluency"] for s in scores]
    amostras_df["geval_coherence"] = [s["coherence"] for s in scores]
    amostras_df["geval_consistency"] = [s["consistency"] for s in scores]
    amostras_df["geval_relevance"] = [s["relevance"] for s in scores]

    return amostras_df

In [ ]:
# =======================================================================
# CÉLULA 10 — Inferência baseline
# =======================================================================
print("=========================================================")
print("INFERÊNCIA BASELINE MediaSum v2 (com Speaker: fala)")
print("=========================================================")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model_baseline = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa"
)
model_baseline.eval()

dialogos, refs, gerados_baseline = gerar_resumos(
    model_baseline, tokenizer, test_set, modo="baseline"
)

df_baseline = pd.DataFrame({
    "dialogo": dialogos,
    "resumo_referencia": refs,
    "resumo_gerado": gerados_baseline
})
df_baseline.to_csv(f"{OUTPUT_DIR}/resultados_baseline.csv", index=False)
print(f"Inferência salva: {len(df_baseline)} resumos")

del model_baseline
torch.cuda.empty_cache()

INFERÊNCIA BASELINE MediaSum v2 (com Speaker: fala)


Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

Gerando resumos (baseline): 100%|██████████| 300/300 [52:21<00:00, 10.47s/it]

Inferência salva: 300 resumos


In [ ]:
# =======================================================================
# CÉLULA 10A — Verificação de alucinações (5 exemplos)
# =======================================================================
df = pd.read_csv(f"{OUTPUT_DIR}/resultados_baseline.csv")
for i, row in df.head(5).iterrows():
    print(f"\n--- Exemplo {i+1} ---")
    print(f"TRANSCRIÇÃO (primeiros 300 chars):\n{row['dialogo'][:300]}...")
    print(f"\nREFERÊNCIA:\n{row['resumo_referencia']}")
    print(f"\nGERADO:\n{row['resumo_gerado']}")
    print("="*60)


--- Exemplo 1 ---
TRANSCRIÇÃO (primeiros 300 chars):
M. O'BRIEN: You know, when I played "Monopoly," I never liked having the utilities. Did you like having the utilities?
ANDY SERWER, "FORTUNE": Steady dividend. Steady.
M. O'BRIEN: It would apparently be a good buy right now, right, Andy Serwer?
SERWER: Absolutely. I'll tell you what's going on right...

REFERÊNCIA:
Utility Usage High Due to Heat

GERADO:
Utility Stocks Soar Amid Heat Wave Demand

--- Exemplo 2 ---
TRANSCRIÇÃO (primeiros 300 chars):
VAUSE: Welcome back, everybody. A short time ago, we got word that tennis executive, Raymond Moore, will step down as CEO of the BNP Paribas Open in California. That decision came after he made statements that stunned many people in and out of the sport.
RAYMOND MOORE, CEO, BNP PARIBAS OPEN: They ri...

REFERÊNCIA:
Tennis Executive Steps Down after Stunning Remarks.

GERADO:
Tennis Executive Resigns Amid Controversial Comments on Gender Pay Disparity

--- Exemplo 3 ---
TRANSCRIÇÃO (primei

In [ ]:
# =======================================================================
# CÉLULA 10B — Backup provisório da inferência
# =======================================================================
import shutil
from google.colab import files

shutil.make_archive("./backup_inferencia_baseline_mediasum_v2", "zip", OUTPUT_DIR, "resultados_baseline.csv")
print(f"Tamanho: {os.path.getsize('./backup_inferencia_baseline_mediasum_v2.zip') / (1024**2):.1f} MB")
files.download("./backup_inferencia_baseline_mediasum_v2.zip")

Tamanho: 0.6 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =======================================================================
# CÉLULA 11 — Métricas e seleção das 40 amostras
# =======================================================================
if 'df_baseline' not in dir():
    print("Carregando inferência do disco...")
    df_baseline = pd.read_csv(f"{OUTPUT_DIR}/resultados_baseline.csv")
    refs = df_baseline["resumo_referencia"].fillna("").astype(str).tolist()
    gerados_baseline = df_baseline["resumo_gerado"].fillna("").astype(str).tolist()
    print(f"-> {len(df_baseline)} resumos carregados!")

df_baseline["resumo_gerado"] = df_baseline["resumo_gerado"].fillna("").astype(str)
refs = df_baseline["resumo_referencia"].fillna("").astype(str).tolist()
gerados_baseline = df_baseline["resumo_gerado"].tolist()

metricas_baseline = calcular_metricas(refs, gerados_baseline, prefixo="Baseline MediaSum v2")

pd.DataFrame([metricas_baseline], index=["Baseline MediaSum v2"]).to_csv(
    f"{OUTPUT_DIR}/metricas_baseline.csv"
)
print(pd.DataFrame([metricas_baseline], index=["Baseline MediaSum v2"]).to_string())

def calcular_rouge_l_individual(row):
    result = rouge.compute(
        predictions=[row["resumo_gerado"]],
        references=[row["resumo_referencia"]]
    )
    return result["rougeL"]

print("\nCalculando ROUGE-L individual...")
df_baseline["rouge_l_individual"] = df_baseline.apply(calcular_rouge_l_individual, axis=1)

amostras_baseline = selecionar_amostras_estratificadas(df_baseline, n=N_AMOSTRAS_MANUAL)
indices_baseline = amostras_baseline.index.tolist()
print(f"Índices selecionados: {indices_baseline}")

pd.DataFrame({"indices": indices_baseline}).to_csv(
    f"{OUTPUT_DIR}/indices_baseline.csv", index=False
)

for col in ["human_fluency", "human_coherence", "human_consistency", "human_relevance",
            "geval_fluency", "geval_coherence", "geval_consistency", "geval_relevance"]:
    amostras_baseline[col] = ""

amostras_baseline.to_csv(f"{OUTPUT_DIR}/amostras_baseline_avaliacao_humana.csv", index=False)
print(f"Amostras e índices salvos com sucesso!")


Calculando métricas Baseline MediaSum v2...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
BERTScore: 100%|██████████| 300/300 [00:10<00:00, 29.32it/s]


                      ROUGE-1  ROUGE-2  ROUGE-L   BLEU  METEOR  BERTScore-P  BERTScore-R  BERTScore-F1
Baseline MediaSum v2   0.2033   0.0569   0.1722  0.017  0.1519       0.9688       0.9717        0.9702

Calculando ROUGE-L individual...
Índices selecionados: [122, 145, 170, 283, 44, 73, 230, 253, 233, 22, 173, 0, 183, 239, 210, 10, 144, 130, 83, 280, 39, 30, 53, 245, 110, 70, 244, 218, 45, 200, 118, 222, 18, 100, 139, 180, 80, 153, 270, 273]
Amostras e índices salvos com sucesso!


In [ ]:
# =======================================================================
# CÉLULA 12 — Verificação dos arquivos salvos
# =======================================================================
print("=== Arquivos na pasta ===")
for f in os.listdir(OUTPUT_DIR):
    tamanho = os.path.getsize(f"{OUTPUT_DIR}/{f}") / 1024
    print(f"{f} — {tamanho:.1f} KB")

print("\n=== metricas_baseline.csv ===")
metricas = pd.read_csv(f"{OUTPUT_DIR}/metricas_baseline.csv", index_col=0)
print(metricas.to_string())

print("\n=== indices_baseline.csv ===")
indices = pd.read_csv(f"{OUTPUT_DIR}/indices_baseline.csv")
print(f"Total de índices: {len(indices)}")
print(indices.head(10))

print("\n=== amostras_baseline_avaliacao_humana.csv ===")
amostras = pd.read_csv(f"{OUTPUT_DIR}/amostras_baseline_avaliacao_humana.csv")
print(f"Linhas: {len(amostras)}")
print(f"Colunas: {amostras.columns.tolist()}")

=== Arquivos na pasta ===
indices_baseline.csv — 0.2 KB
resultados_baseline.csv — 1530.0 KB
amostras_baseline_avaliacao_humana.csv — 220.3 KB
metricas_baseline.csv — 0.1 KB

=== metricas_baseline.csv ===
                      ROUGE-1  ROUGE-2  ROUGE-L   BLEU  METEOR  BERTScore-P  BERTScore-R  BERTScore-F1
Baseline MediaSum v2   0.2033   0.0569   0.1722  0.017  0.1519       0.9688       0.9717        0.9702

=== indices_baseline.csv ===
Total de índices: 40
   indices
0      122
1      145
2      170
3      283
4       44
5       73
6      230
7      253
8      233
9       22

=== amostras_baseline_avaliacao_humana.csv ===
Linhas: 40
Colunas: ['dialogo', 'resumo_referencia', 'resumo_gerado', 'rouge_l_individual', 'human_fluency', 'human_coherence', 'human_consistency', 'human_relevance', 'geval_fluency', 'geval_coherence', 'geval_consistency', 'geval_relevance']


In [ ]:
# =======================================================================
# CÉLULA 13 — Backup completo do baseline
# =======================================================================
import shutil
from google.colab import files

os.makedirs("./backup_baseline_mediasum_v2", exist_ok=True)
shutil.copytree(OUTPUT_DIR, "./backup_baseline_mediasum_v2/avaliacao_mediasum_v2", dirs_exist_ok=True)
shutil.make_archive("./backup_baseline_mediasum_v2", "zip", "./backup_baseline_mediasum_v2")
print(f"Backup gerado: backup_baseline_mediasum_v2.zip ({os.path.getsize('./backup_baseline_mediasum_v2.zip') / (1024**2):.1f} MB)")
files.download("./backup_baseline_mediasum_v2.zip")

Backup gerado: backup_baseline_mediasum_v2.zip (0.7 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>